# 1G Repeater Network Theoretical Rate Simulation

This example will demonstrate how to use QNPack to simulate a quantum network repeater chain using a multiple quantum repeaters (QRs) and calculate theoretically the entanglement rate.
We present a theoretical rate calculation that accounts for all sources of photon loss across the quantum network, including losses introduced by the trap-cavity system, fiber transmission, detectors, and other optical components. This model offers a comprehensive framework for calculating the entanglement generation rate for arbitrary numbers of repeaters, entanglement attempts, and varying photon loss parameters. By systematically incorporating each loss mechanism, we derive an analytical expression that quantifies the achievable entanglement rate as a function of key network parameters, enabling performance prediction across different configurations.

In our formulation, the following notations are used:

- $P_f$ — Total probability of successful BSM at all BSM-nodes in the first step of the routing protocol governed by the control node  
- $P_l$ — Total probability of successful BSM at all BSM-nodes in the second (final) step of the routing protocol  
- $P_{f,i}(s_i)$ — Probability of success at attempt $s_i$ for BSM-nodes in the first step  
- $P_{l,j}(s_j)$ — Probability of success at attempt $s_j$ for BSM-nodes in the second step  
- $retries$ — Maximum number of entanglement attempts allowed  
- $s$ — Attempt at which BSM was successful  
- $T_{\text{avg}}$ — Average time required per repeater cycle  
- $T_{\text{retry}}$ — Time required for one entanglement attempt  
- $T_{\text{bsm}}$ — BSM time with correction  
- $T_{\text{dbsm}}$ — DBSM time with correction  

$$
T_{\text{avg}} = \sum_{s_i=1}^{y+1} \sum_{s_j=1}^{y+1} 
\left[ \prod_{i} \prod_{j} P_{f,i}(s_i) P_{l,j}(s_j) \right] T_{\text{retry}} 
\cdot \left( \max(s_i) + \max(s_j) \right) \\
+ \left(1 - P_f\right) \cdot T_{\text{retry}} \cdot (y+1) \\
+ \sum_{s_i=1}^{y+1} \prod_{i} P_{f,i}(s_i) \left(1 - P_l\right) 
\left[ T_{\text{retry}} \cdot \max(s_i) + T_{\text{retry}} \cdot (y+1) \right]
$$

In simulations, at shorter distances, the entanglement generation rate tends to decrease as more repeaters are added. This is because repeaters are primarily beneficial for extending entanglement over long distances; at short distances, the total time required for entanglement likecontrol delays and retries; that can exceed that of direct transmission, leading to reduced overall efficiency.

Theoretical calculations rely on probabilistic models that do not explicitly account for entanglement failures. As a result, they tend to overestimate the rate, especially when fewer retries are allowed. The apparent increase in rate with fewer retries is more pronounced in theory than in simulation because simulations more accurately capture practical limitations such as failed entanglement attempts and synchronization delays—factors that are often omitted in theoretical models. 

In [ ]:
from qnpack.oneG.theo_rate import TheoRateSimulation
fixed_params = {
    "network": {"col_eff": 0.70},
    "ion_trap": {"retries": 30},
}

varying_params = {
    "num_repeaters": [2, 4, 8],
    "distances": [10],
}

sim = TheoRateSimulation(fixed_params=fixed_params, varying_params=varying_params,
                        parameter_file="parameters/theo_rate.yml",
                        output_dir="results")
data = sim.start()
sim.plot(final_data=data, x_axis1="num_repeaters", y_axis1="rate",  xlabel1="Number of Repeaters", ylabel1="Theoretical Rate",
         label_param1="Distance", title1="Fidelity vs Number of Repeaters")

Updated network.col_eff: 0.69 -> 0.7
Updated ion_trap.retries: 90 -> 30
02:11:20 qnpack.oneG.theo_rate INFO Running theoretical rate calculation with parameters: {'num_repeaters': [2, 4, 8], 'distances': [10]}
02:11:20 qnpack.oneG.theo_rate INFO Fixed parameters: {'network': {'col_eff': 0.7}, 'ion_trap': {'retries': 30}}
02:11:20 qnpack.oneG.theo_rate INFO Running theoretical rate calculation with parameters: {'num_repeaters': 2, 'distances': 10}
Rate for 10 with 2 repeaters is: 69.08837561695115
02:11:21 qnpack.oneG.theo_rate INFO Running theoretical rate calculation with parameters: {'num_repeaters': 4, 'distances': 10}
Rate for 10 with 4 repeaters is: 70.0907440466042
02:12:06 qnpack.oneG.theo_rate INFO Running theoretical rate calculation with parameters: {'num_repeaters': 8, 'distances': 10}


### Comparison with real simulation

In [ ]:
from qnpack.oneG.iontrap import IonTrapSimulation
fixed_params = {
    "photon_loss": 0.2,
    "init_photon_loss": 0.7,
    "coherence_time": 60000000, #ns
}

varying_params = {
    "num_repeaters": [2, 4, 6, 8],
    "distance": [50],
}

sim = IonTrapSimulation(fixed_params=fixed_params, varying_params=varying_params,
                        iterations=10,
                        parameter_file="parameters/noisy.yml",
                        output_dir="results")
data = sim.start()
sim.plot(final_data=data, x_axis1="num_repeaters", y_axis1="fidelity",  xlabel1="Number of Repeaters", ylabel1="Fidelity",
         label_param1="distance", title1="Fidelity vs Number of Repeaters",
         x_axis2="num_repeaters", y_axis2="rate",  xlabel2="Number of Repeaters", ylabel2="Rate (Hz)",
         label_param2="distance", title2="Rate vs Number of Repeaters")